In [ ]:
# Problema: Mostrar cómo una clave sesgada concentra trabajo en una partición y cómo un combiner reduce pares antes del shuffle.

import csv
import hashlib
from collections import defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data").is_dir() and (p / "submission").is_dir())
with (ROOT / "data/events.csv").open(encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))
def partition(key): return int(hashlib.md5(key.encode()).hexdigest(), 16) % 4

In [ ]:
# event_id reparte filas; account_id reúne eventos relacionados y evidencia la clave HOT.

balanced, skewed = [0] * 4, [0] * 4
for row in rows:
    balanced[partition(row["event_id"])] += 1
    skewed[partition(row["account_id"])] += 1
balanced, skewed

In [ ]:
# El combiner suma cada cuenta dentro de su partición antes de intercambiar pares.

partials = [defaultdict(float) for _ in range(4)]
for row in rows:
    partials[partition(row["account_id"])][row["account_id"]] += float(row["amount"])
combined_pairs = sum(len(item) for item in partials)
combined_pairs

In [ ]:
# El resultado hace visible carga, sesgo y reducción de pares.

with (ROOT / "submission/partition_loads.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["partition", "event_key_load", "account_key_load"], lineterminator="\n")
    writer.writeheader(); writer.writerows({"partition": i, "event_key_load": balanced[i], "account_key_load": skewed[i]} for i in range(4))
with (ROOT / "submission/shuffle_comparison.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["raw_pairs", "pairs_after_local_combiner", "hot_key_load"], lineterminator="\n")
    writer.writeheader(); writer.writerow({"raw_pairs": len(rows), "pairs_after_local_combiner": combined_pairs, "hot_key_load": max(skewed)})